# M51 · Mechanisms from first motions: polarity misfit, grid search, Kagan angle and SKHASH

**Part 3 · week 6 · about 10 h core**

## 한국어 요약

이 모듈에서는 P파 초동 극성(위 또는 아래)으로 지진의 단층면해를 구합니다. 먼저 주향, 경사, 미끄럼각이 주어졌을 때 각 관측소에서 예상되는 초동 극성을 계산하고 관측 극성과의 가중 불일치를 재는 함수, 주향-경사-미끄럼각 격자를 모두 훑어 가장 잘 맞는 단층면해와 허용 가능한 단층면해 집합을 찾는 함수, 두 단층면해 사이의 차이를 재는 카간 각 함수, 허용 가능한 단층면해를 서로 다른 무리로 나누는 함수를 정답을 아는 합성 자료로 작성하고 점검합니다. 이어서 PocketQuake로 2018-2022년 상주 지진 여섯 개에 SKHASH를 실행하고, 같은 극성 자료에 여러분의 격자 탐색을 적용해 SKHASH의 결과 및 한국지질자원연구원 지진연보의 본진 단층면해와 비교합니다. 마지막으로 아무것도 바꾸지 않은 재실행과 설정 하나만 바꾼 실행을 비교하면서, PocketQuake가 SKHASH에 넘기는 S/P 진폭비가 무엇을 담고 있는지 직접 측정하고 설명합니다.

## What you will be able to do

- Predict the first-motion polarities of a double-couple mechanism at stations of known azimuth and takeoff angle, and score the mechanism by its weighted polarity misfit
- Find the best mechanism and the set of acceptable mechanisms with a grid search over strike, dip and rake, and test it on made-up polarities with some readings flipped
- Measure the difference between two mechanisms with the Kagan angle, and group acceptable mechanisms into families
- Run SKHASH through PocketQuake on the Sangju earthquakes and compare its mechanisms with your grid search on the same polarities and with the KIGAM bulletin mechanism of the mainshock
- Rerun SKHASH unchanged and then with one setting changed, measure what the S/P amplitude ratios that PocketQuake hands to SKHASH contain, and explain how they affect the mechanisms and their quality grades

## Before you start

- Modules: M21, M32, M50
- Reading: Aki & Richards (2002), box 4.4 (fault normal, slip vector and the radiation of P waves); Hardebeck & Shearer (2002), the sections on the method (acceptable mechanisms, the multiple solutions and the quality grades); Skoumal, Hardebeck & Shearer (2024), the description of the inputs, the outputs and the quality grades; Kagan (1991), the definition of the rotation between two double couples (a skim is enough)
- Needs: synthetic, stp, pocketquake
- Key terms (see the glossary): focal mechanism (단층면해, 진원 메커니즘), strike (주향), dip (경사), rake (미끄럼각, 레이크), fault normal (단층면 법선 벡터), slip vector (미끄러짐 벡터), double couple (이중 우력, 더블 커플), radiation pattern (방사 패턴), first-motion polarity (초동 극성), ray direction (파선 방향), azimuth (방위각), takeoff angle (이탈각), focal sphere (진원구), lower-hemisphere projection (하반구 투영), beachball (비치볼, 단층면해 그림), nodal plane (절면), auxiliary plane (보조면), P and T axes (P축과 T축), misfit function (목적 함수, 불일치 함수), grid search (격자 탐색), acceptable mechanism (허용 가능한 단층면해), Kagan angle (카간 각), rotation matrix (회전 행렬), rotation space (회전 공간), mechanism family (단층면해 무리), SKHASH (단층면해 결정 프로그램), cuspid (cuspid (SKHASH 지진 번호)), mechanism quality (단층면해 품질 등급), S/P amplitude ratio (S/P 진폭비), amplitude window (진폭 측정 구간), S-P time (S-P 시간), PocketQuake (지진 분석 도구 모음), cluster (PocketQuake; 클러스터 (PocketQuake)), PhaseNet+ (PhaseNet+), pick probability (피킹 확률), HYPOINVERSE (진원 결정 프로그램), velocity model (속도 구조 모델), dry run (드라이 런, 시험 실행), checksum (SHA-256 fingerprint; 체크섬), run log (실행 기록), deterministic computation (결정론적 계산), widget (위젯)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M51.yaml](../../../.sgtl/criteria/M51.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M51.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
PQ_SLUG = "sangju_m21"                    # your PocketQuake cluster of module M21, located in module M32
PQ_CORES = 1                              # cores of the mechanism stage (SKHASH runs on one)
PQ_MEM_GB = 12.0                          # memory cap of each run in GB (SKHASH with S/P ratios needs about 9)
VELMODEL = "kim1983"                      # the velocity model of PocketQuake's mechanism stage (fm_velmodel)
MIN_WEIGHT = 0.3                          # Part 5: smallest polarity size used (fm_min_polarity_weight)
STEP_DEG = 5.0                            # Part 5: grid step of your grid search on the Sangju polarities
TOL = 0.1                                 # Part 5: misfit tolerance of the acceptable mechanisms
FAMILY_DEG = 45.0                         # Part 5: Kagan angle that separates two families (SKHASH's cangle)
KIGAM_MAINSHOCK = (288.0, 89.0, 7.0)      # Part 5: KIGAM's mechanism of the 2019-07-21 ML 3.9 (one nodal plane)
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M51")

import contextlib
import hashlib
import io
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Image, display


def show_figure(figure, filename):
    """Save and display the same static figure, including in an archived notebook."""
    path = Path("figures") / filename
    path.parent.mkdir(exist_ok=True)
    figure.savefig(path, dpi=150, bbox_inches="tight")
    display(Image(filename=str(path)))
    plt.close(figure)

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
RUN_DIR = st.pq.pipeline_dir() / "pipeline" / "runs" / PQ_SLUG      # PocketQuake's results for your cluster
FM_DIR = RUN_DIR / "3.FocalMech" / VELMODEL                         # what its mechanism stage writes
print("PocketQuake results of your cluster:", RUN_DIR)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M51-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up data whose answers are known from how they
were built (`st.reroll("M51-T1")` draws new ones). Tasks 5 and 6 also read products of your own PocketQuake runs
and the run log; the checks never print the values they compare with.

The order of the work is the one of the earlier modules. Tasks 1-4 write the core functions and test them on
made-up polarities with a known answer. Part 5 runs SKHASH, the lab's mechanism program, through PocketQuake on the
six Sangju earthquakes and applies your functions to the same polarities, so that you can compare your mechanisms
with an independent program and with a published one. Part 6 runs the program again: unchanged first, then with
one setting changed.

This module needs your STP download `sangju2019` (module M03), the course tools (`bin/sgtl-install tools`), and
your PocketQuake cluster located in module M32 (`PQ_SLUG`). Parts 1-4 need none of them. One run of the mechanism
stage takes about a minute on one core and up to about 9 GB of memory.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M51.md`.

## Part 0 · First motions and the focal sphere (about 30 min)

A **focal mechanism** (glossary `focal_mechanism`) gives the orientation of a fault and the direction of slip on
it. Module M50 described it with three angles: the **strike** (glossary `strike`) φ, the compass direction of a
horizontal line in the fault plane, chosen so that the fault dips to the right; the **dip** (glossary `dip`) δ,
from 0 to 90 degrees below the horizontal; and the **rake** (glossary `rake`) λ, the direction of slip of the upper
side (the hanging wall) measured in the fault plane from the strike direction. In axes pointing north, east and
down, two unit vectors follow from them (Aki & Richards 2002, box 4.4), and you wrote the function that makes them
in module M50 (task M50-T1): the **fault normal** n (glossary `fault_normal`), which points up from the footwall
into the hanging wall, and the **slip vector** d (glossary `slip_vector`), the direction in which the hanging wall
moves. This module uses your own `fault_vectors`: paste it into the next cell as it is in your M50 notebook (it
needs numpy only) and run it. Task 1 may call it, and the cells after the check of task 1 use it.

In [ ]:
# Paste your fault_vectors of module M50 below this line.

Slip on a flat fault radiates like a **double couple** (glossary `double_couple`). Its P-wave **radiation pattern**
(glossary `radiation_pattern`) in the direction of a unit vector **r** leaving the source is

$$A_P(\mathbf{r}) \propto 2\,(\mathbf{r}\cdot\mathbf{n})\,(\mathbf{r}\cdot\mathbf{d}).$$

Where $A_P > 0$ the first P motion pushes the ground away from the source: a seismometer's vertical component
first moves up (compression); where $A_P < 0$ it first moves down (dilatation). This is the first-motion
**polarity** (glossary `polarity`) that you read by hand in module M20 and that PhaseNet+ reads in module M21. A
polarity is read from the counts: in module M03 (Part 6) you saw that a negative stage gain in the station metadata
(StationMetadata 1.0.2 by Byeong-Seok Ahn, Pukyong National University) is not a reversed channel, so no polarity
here is ever flipped by a metadata sign.

The **ray direction** (glossary `ray_direction`) at the source follows from two angles: the **azimuth** (glossary
`azimuth`) a of the station, clockwise from north, and the **takeoff angle** (glossary `takeoff_angle`) i, measured
here from the downward vertical (0 = straight down, 90 = horizontal, 180 = straight up; module M30 traced such
rays):

$$\mathbf{r} = (\sin i\,\cos a,\;\; \sin i\,\sin a,\;\; \cos i)\qquad\text{(north, east, down)}.$$

Picture all ray directions as points on a small sphere around the source, the **focal sphere** (glossary
`focal_sphere`). $A_P$ is zero on two perpendicular planes through the source, the **nodal planes** (glossary
`nodal_plane`): the fault plane and the **auxiliary plane** (glossary `auxiliary_plane`), whose normal is the slip
vector. They cut the sphere into two compressional and two dilatational quadrants. Halfway between the planes lie
the **P and T axes** (glossary `p_t_axes`): $\mathbf{T} = (\mathbf{n}+\mathbf{d})/\sqrt2$ in the middle of the
compressional quadrants and $\mathbf{P} = (\mathbf{n}-\mathbf{d})/\sqrt2$ in the middle of the dilatational ones;
$\mathbf{B} = \mathbf{T}\times\mathbf{P}$ lies along both planes. Drawn in a **lower-hemisphere projection**
(glossary `lower_hemisphere_projection`), with compressions shaded, the sphere becomes the **beachball** (glossary
`beachball`) of module M50. A first-motion mechanism is the inverse problem: find the planes that put every
observed up in a compressional quadrant and every down in a dilatational one (Reasenberg & Oppenheimer 1985; Snoke
2003; Hardebeck & Shearer 2002; Shearer 2019, chapter 9; Stein & Wysession 2003, section 4.2).

The six earthquakes of your download, from the KMA catalog that ships with the course (UTC):

In [ ]:
SAC_ROOT = st.data.root() / DATASET / "stp_download" / "SAC"
kma = st.data.sequence("sangju2019")
folders = sorted(p.name for p in SAC_ROOT.iterdir() if p.is_dir()) if SAC_ROOT.is_dir() else []
six = kma[kma["origin_utc"].dt.strftime("%Y%m%d%H%M%S").isin(folders)].reset_index(drop=True)
print(f"{len(folders)} earthquakes in your download {DATASET!r}; in the KMA catalog subset:")
print(six[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]].to_string(index=False))

## Part 1 · The polarities a mechanism predicts (about 20 min, then task 1)

A **misfit function** (glossary `misfit_function`) turns "how badly does this mechanism fit?" into one number. For
first motions the simplest one counts the stations whose observed polarity differs from the predicted one. Not all
readings are equally sure: PhaseNet+ gives each polarity a size from 0 to 1 (module M21), and a sure reading
should count more than a doubtful one. So each station k gets a weight $w_k$, and the misfit is the weighted share
of mismatches,

$$F = \frac{\sum_k w_k\,[\,p_k \ne \operatorname{sign} A_P(\mathbf{r}_k)\,]}{\sum_k w_k},$$

where $p_k = \pm1$ is the observed polarity and $[\cdot]$ is 1 when true and 0 when false. $F = 0$ fits every
station; $F = 0.1$ means a tenth of the weight disagrees.

### M51-T1 · The polarities a mechanism predicts, and its misfit · core · about 60 min

**Goal:** write `polarity_misfit(strike, dip, rake, azimuth, takeoff, polarity, weight)`: the polarity that a
mechanism predicts at every station, and its weighted misfit.

**Given:** Part 0 (your `fault_vectors` of M50 for n and d, the ray r and the amplitude $A_P$) and Part 1 (the
misfit). numpy only. Module M50 built the two vectors (task 1) and predicted the first motions of one mechanism
(task 6); this function adds weights and handles many mechanisms at once, as the grid search of task 2 needs. You
may call your `fault_vectors` (Part 0) and your other M50 functions inside it.

**Contract:**
- Inputs: `strike`, `dip`, `rake` in degrees, either three numbers (one mechanism) or three 1-D arrays of the same
  length M (M mechanisms); `azimuth` (degrees clockwise from north), `takeoff` (degrees from the downward vertical,
  0 to 180), `polarity` (+1 up, -1 down) and `weight` (at least 0, not all 0): 1-D numpy arrays of one value per
  station, N stations. Do not change them.
- Output: a tuple `(misfit, predicted)`. For one mechanism, `misfit` is a number and `predicted` an array of N
  values; for M mechanisms, `misfit` has shape (M,) and `predicted` shape (M, N). `predicted` is the sign of $A_P$
  (+1, -1, or 0 exactly on a nodal plane, which counts as a mismatch); `misfit` is $F$ of Part 1, from 0 to 1.
- Example: strike 0, dip 90, rake 0 (a vertical fault striking north, slipping horizontally) and horizontal rays
  (takeoff 90) to azimuths 45, 135, 225 and 315 predict +1, -1, +1, -1; with observed polarities +1, +1, +1, -1 and
  weights 1, 0.5, 1, 1, the misfit is 0.5 / 3.5 = 0.142857.

**Done when** `st.check("M51-T1", func=polarity_misfit)` passes on 16 made-up cases (one mechanism and several,
rays going down and up, some polarities flipped, equal and unequal weights).

**Stuck?** `st.hint("M51-T1", 1)`

In [ ]:
# Before you code: take the mechanism of the example (strike 0, dip 90, rake 0). Which polarity does it predict at a
# station due north of the source, with a horizontal ray? At a station far to the north-east whose ray leaves the
# source steeply upward (takeoff 150)? And if every observed polarity of a data set were read upside down, what
# would happen to the misfit of every mechanism?
st.predict("M51-T1", """

""")

In [ ]:
def polarity_misfit(strike, dip, rake, azimuth, takeoff, polarity, weight):
    """Predicted P polarities of one or several mechanisms, and their weighted polarity misfit.

    strike, dip, rake : degrees; three numbers, or three 1-D arrays of M values (M mechanisms)
    azimuth, takeoff  : degrees; azimuth clockwise from north, takeoff from the downward vertical (N stations)
    polarity, weight  : +1 (up) or -1 (down), and the weight of each station (not changed)
    returns           : (misfit, predicted): a number and N signs, or M misfits and an (M, N) array of signs
    """
    # Step 1: the angles in radians; for M mechanisms give each angle a second axis so that numpy broadcasts the
    #         M mechanisms against the N stations.
    # Step 2: the fault normal n and the slip vector d (north, east, down) as your fault_vectors of M50 builds them
    #         (Part 0), for every mechanism at once, and the ray r of each station.
    # Step 3: A_P = 2 (r . n)(r . d), component by component; predicted = its sign.
    # Step 4: misfit = summed weight where predicted differs from polarity, divided by the summed weight.
    # YOUR CODE HERE
    raise NotImplementedError("M51-T1: write your code here")

In [ ]:
st.check("M51-T1", func=polarity_misfit)

Your function on the example of the Contract, next to the signs that your M50 vectors give on the same rays; then
on the two stations of the prediction, and on the example with every polarity flipped:

In [ ]:
ex_az = np.array([45.0, 135.0, 225.0, 315.0])
ex_inc = np.full(4, 90.0)
ex_pol = np.array([1, 1, 1, -1])
ex_w = np.array([1.0, 0.5, 1.0, 1.0])
print("example:", polarity_misfit(0.0, 90.0, 0.0, ex_az, ex_inc, ex_pol, ex_w))
if not callable(globals().get("fault_vectors")):
    raise NameError("fault_vectors not found: paste your function of module M50 into the cell of Part 0 and run it "
                    "(finish task M50-T1 of module M50 first).")
n_ex, d_ex = fault_vectors(0.0, 90.0, 0.0)                     # your M50 function (Part 0)
ray_ex = np.column_stack([np.sin(np.radians(ex_inc)) * np.cos(np.radians(ex_az)),
                          np.sin(np.radians(ex_inc)) * np.sin(np.radians(ex_az)), np.cos(np.radians(ex_inc))])
print("your M50 vectors of 0/90/0: n", (np.round(n_ex, 3) + 0.0).tolist(), " d", (np.round(d_ex, 3) + 0.0).tolist(),
      "; sign of 2 (r . n)(r . d) on the example rays:", np.sign(2.0 * (ray_ex @ n_ex) * (ray_ex @ d_ex)).astype(int))
print("due north, horizontal; north-east, takeoff 150:",
      polarity_misfit(0.0, 90.0, 0.0, np.array([0.0, 45.0]), np.array([90.0, 150.0]), np.ones(2), np.ones(2))[1])
print("every polarity flipped:", polarity_misfit(0.0, 90.0, 0.0, ex_az, ex_inc, -ex_pol, ex_w)[0])
three = polarity_misfit(np.array([0.0, 0.0, 90.0]), np.array([90.0, 90.0, 90.0]), np.array([0.0, 180.0, 0.0]),
                        ex_az, ex_inc, ex_pol, ex_w)
print("three mechanisms (0/90/0, 0/90/180, 90/90/0), misfits:", np.round(three[0], 6))
print(three[1])

**Explain (3-5 sentences):** Use the printed lines. What did your function predict due north, and is that number
physics or rounding (what is cos 90 degrees in floating point)? What does that say about a station on or near a
nodal plane? Which two of the three mechanisms in the last line predict opposite polarities everywhere, and which two
the same, and what does that say about rake 0 and rake 180 on one plane, and about the auxiliary plane? Why do the
misfits of the example with every polarity flipped and without add up to 1?

## Part 2 · A grid search over strike, dip and rake (about 20 min, then task 2)

A **grid search** (glossary `grid_search`; module M31) tries every point of a grid and keeps the best. For
mechanisms the grid runs over strike, dip and rake. Two things make it special:

- every double couple can be written twice, with its fault plane and with its auxiliary plane (the last line of
  Part 1), and first motions alone cannot tell which plane slipped; when both versions lie on the grid, the misfit
  has two equal minima;
- polarities never fix one mechanism exactly: many grid points fit about equally well. HASH (Hardebeck & Shearer
  2002) and its successor **SKHASH** (glossary `skhash`; Skoumal et al. 2024) therefore keep the set of
  **acceptable mechanisms** (glossary `acceptable_mechanism`): all mechanisms whose misfit is close to the
  smallest, with "close" set by the share of polarities assumed wrong. The spread of that set says how well the
  mechanism is known.

Here an acceptable mechanism is one whose misfit is at most the best misfit plus a tolerance `tol` (a share of the
total weight). SKHASH's own rule (read in Part 5) uses a count of polarities instead, and repeats the search with
the station angles shaken within their errors.

### M51-T2 · A grid search over strike, dip and rake · core · about 60 min

**Goal:** write `grid_search(azimuth, takeoff, polarity, weight, step_deg, tol, misfit)`: the best mechanism on a
regular grid, and the acceptable mechanisms.

**Given:** your `polarity_misfit` of task 1 (the check hands in a misfit function of its own with the same contract,
so that this task is tested on its own). numpy only.

**Contract:**
- Inputs: the four station arrays of task 1 (not changed); `step_deg`, a grid step that divides 90 (for example 5,
  10, 15, 18 or 30); `tol` (at least 0); `misfit`, a function with the contract of task 1.
- The grid: strikes 0, step, 2 step, ... below 360; dips step, 2 step, ... up to 90 (no dip 0: a horizontal plane
  has no strike); rakes -180 + step, ... up to 180. Its order: strike changes slowest, rake fastest. Ask `misfit`
  about every grid point once (one call with arrays is fastest).
- Output: a tuple `(best, best_misfit, accepted, accepted_misfit)`: `best` = (strike, dip, rake) of the first grid
  point (in the grid's order) with the smallest misfit, `best_misfit` its misfit; `accepted`, an array of shape
  (K, 3) of every grid point whose misfit is at most `best_misfit + tol`, sorted by misfit (equal misfits in the
  grid's order); `accepted_misfit`, their K misfits.
- Example: with `step_deg = 30` the grid has 12 strikes, 3 dips and 12 rakes, 432 points; its first point is
  (0, 30, -150) and its last (330, 90, 180).

**Done when** `st.check("M51-T2", func=grid_search)` passes on 12 made-up cases (30-60 stations around a made-up
mechanism, up to a tenth of the polarities flipped, steps of 10-30 degrees, `tol` 0, 0.05 or 0.1), and the median
Kagan angle (task 3) between your best mechanisms and the made-up ones stays small.

**Stuck?** `st.hint("M51-T2", 1)`

In [ ]:
# Before you code: 40 stations spread around a made-up mechanism, 4 of their polarities flipped, a grid of 10-degree
# steps. Roughly how many grid mechanisms will share the smallest misfit: one, two, a handful, hundreds? Will the
# best misfit be 0, about 4/40, or more? How large is the accepted set likely to be with tol = 0.05?
st.predict("M51-T2", """

""")

In [ ]:
def grid_search(azimuth, takeoff, polarity, weight, step_deg, tol, misfit):
    """The best mechanism on a strike-dip-rake grid, and the acceptable mechanisms.

    azimuth, takeoff, polarity, weight : the station arrays of task 1 (not changed)
    step_deg : grid step in degrees (divides 90); tol : misfit tolerance of the acceptable set
    misfit   : a function with the contract of task 1
    returns  : (best (strike, dip, rake), best_misfit, accepted (K, 3), accepted_misfit (K,))
    """
    # Step 1: the three ranges of the Contract, and every (strike, dip, rake) triple with strike changing slowest.
    # Step 2: one call of misfit with the three columns of the grid.
    # Step 3: the first smallest value; then every point within tol of it, sorted by misfit with a stable sort.
    # YOUR CODE HERE
    raise NotImplementedError("M51-T2: write your code here")

In [ ]:
st.check("M51-T2", func=grid_search)

Your grid search on a made-up case like the one of the prediction: 40 stations around a made-up mechanism, 4
polarities flipped (the generator below is the notebook's own, with a fixed seed).

In [ ]:
practice_rng = np.random.default_rng(51)
TRUE_MECH = (120.0, 60.0, 45.0)                         # a made-up oblique reverse mechanism
p_az = practice_rng.uniform(0.0, 360.0, 40)
p_inc = practice_rng.uniform(20.0, 160.0, 40)
p_pol = polarity_misfit(*TRUE_MECH, p_az, p_inc, np.ones(40), np.ones(40))[1].astype(float)
p_flip = practice_rng.choice(40, 4, replace=False)
p_pol[p_flip] *= -1.0
p_w = np.ones(40)
for p_tol in (0.0, 0.05):
    p_best, p_best_misfit, p_acc, p_acc_misfit = grid_search(p_az, p_inc, p_pol, p_w, 10.0, p_tol, polarity_misfit)
    print(f"tol {p_tol}: best {p_best}, misfit {p_best_misfit:.3f}, {len(p_acc)} acceptable mechanisms "
          f"({np.count_nonzero(p_acc_misfit == p_best_misfit)} share the smallest misfit)")
print("the made-up mechanism:", TRUE_MECH, " its own misfit:",
      round(polarity_misfit(*TRUE_MECH, p_az, p_inc, p_pol, p_w)[0], 3))

**See a mechanism before reading its three angles.** Shading marks compressional P radiation on a lower
hemisphere; white marks dilatation. The last panel places the synthetic observations by azimuth and takeoff
angle. It is an angular coverage plot, not a map. Black rings identify readings flipped by the known experiment.

**Look:** how different can two acceptable beachballs look? Which parts of the source are poorly sampled by
these stations? A best-fitting grid point is one member of an acceptable set, not the full uncertainty.

In [ ]:
from obspy.imaging.beachball import beach


def draw_mechanism(axis, mechanism, title, color="tab:blue"):
    """Plot supplied strike/dip/rake values with ObsPy; no inversion is performed here."""
    axis.add_collection(beach(list(map(float, mechanism)), xy=(0, 0), width=1.7, size=100,
                              facecolor=color, edgecolor="0.2", linewidth=0.7))
    axis.set(xlim=(-1, 1), ylim=(-1, 1), aspect="equal", title=title)
    axis.axis("off")
    axis.text(0, 0.93, "N", ha="center", va="bottom", fontsize=8)


fig, axes = plt.subplots(1, 4, figsize=(12.5, 3), constrained_layout=True)
draw_mechanism(axes[0], TRUE_MECH, "Known synthetic source")
draw_mechanism(axes[1], p_best, f"Your best: misfit {p_best_misfit:.3f}")
draw_mechanism(axes[2], p_acc[-1], f"Accepted alternative: F={p_acc_misfit[-1]:.3f}")
axes[3].scatter(p_az, p_inc, c=p_pol, cmap="bwr", vmin=-1, vmax=1, s=25)
axes[3].scatter(p_az[p_flip], p_inc[p_flip], facecolors="none", edgecolors="k", s=65, label="Flipped observations")
axes[3].set(xlabel="Azimuth clockwise from N (°)", ylabel="Takeoff from downward vertical (°)",
            xlim=(0, 360), ylim=(180, 0), title="Synthetic station coverage")
axes[3].legend(fontsize=7)
show_figure(fig, "M51_synthetic_mechanisms.png")

**Explain (3-5 sentences):** Use the printed lines. How many mechanisms shared the smallest misfit, and why not
the two of Part 2 (is the made-up mechanism, or its auxiliary plane, on a grid of 10-degree steps)? Is the best
misfit lower or higher than the made-up mechanism's own misfit, and why? How many mechanisms became acceptable with
tol = 0.05, and what does that number say about how well 40 polarities fix a mechanism?

## Part 3 · How different are two mechanisms? (about 20 min, then task 3)

Comparing strikes, dips and rakes one by one is misleading: (0, 90, 0) and (90, 90, 180) look entirely different
and are the same double couple (Part 1). The **Kagan angle** (glossary `kagan_angle`; Kagan 1991) is the smallest
rotation that turns one double couple into the other: 0 degrees means the same mechanism, and no two double
couples are more than 120 degrees apart.

Put the T, P and B axes of a mechanism (Part 0; module M50, task 3, gave P and T as trend and plunge, here you
need them as unit vectors) as the three columns of a 3 x 3 matrix A. A **rotation matrix**
(glossary `rotation_matrix`) R turns frame A1 into frame A2 when A2 = A1 R, so $R = A_1^{\mathsf T}A_2$, and the
angle of the rotation follows from its trace: $\cos\theta = (\operatorname{tr}R - 1)/2$. A double couple does not
change when you turn it half a turn about its T, P or B axis (two of its axes flip sign): A2 can be written in four
ways (no flip, or the signs of two of its columns flipped), and the Kagan angle is the smallest of the four
rotation angles (Kagan 1991; Tape & Tape 2012).

A picture helps in Part 4. Think of every orientation that a double couple can take as one point of a **rotation
space** (glossary `rotation_space`; Kagan 1991; Tape & Tape 2012), in which the Kagan angle is the distance between
two points. Turning one mechanism further and further about one fixed axis moves its point along a line in that
space, and two points on the line are as far apart as the angle turned between them (up to 90 degrees; beyond
that one of the four versions can come closer).

### M51-T3 · The Kagan angle between two mechanisms · core · about 45 min

**Goal:** write `kagan_angle(mech1, mech2)`: the Kagan angle between two double couples, in degrees.

**Given:** Part 0 (your `fault_vectors` for n and d), Part 3 (T, P, B, the rotation and its four versions). numpy
only; the libraries that compute Kagan angles are not allowed (the check uses one of them).

**Contract:**
- Inputs: `mech1`, `mech2`: (strike, dip, rake) in degrees, as tuples, lists or arrays of three numbers.
- Output: one number from 0 to 120 (degrees). The same in both orders.
- Example: a vertical strike-slip fault (0, 90, 0) and the same fault turned 30 degrees, (30, 90, 0), are 30
  degrees apart; (0, 90, 0) and (90, 90, 180) are 0 degrees apart.

**Done when** `st.check("M51-T3", func=kagan_angle)` passes on 30 made-up pairs (random pairs, one mechanism written
with either nodal plane, nearby pairs, identical pairs), in both orders.

**Stuck?** `st.hint("M51-T3", 1)`

In [ ]:
# Before you code: what is the Kagan angle between a pure normal fault (strike 0, dip 45, rake -90) and a pure reverse
# fault on the same plane (0, 45, 90)? Between the normal fault and a vertical strike-slip fault (0, 90, 0)? Which of
# the two pairs is farther apart?
st.predict("M51-T3", """

""")

In [ ]:
def kagan_angle(mech1, mech2):
    """The Kagan angle (degrees, 0-120) between two double couples given as (strike, dip, rake) in degrees."""
    # Step 1: for each mechanism the normal n and slip d (Part 0), then T = (n + d)/sqrt 2, P = (n - d)/sqrt 2,
    #         B = T x P as the columns of a 3 x 3 matrix.
    # Step 2: R = A1 transposed times A2; for each of the four sign patterns of the columns of A2, the cosine of the
    #         rotation angle is (trace - 1)/2.
    # Step 3: the largest cosine gives the smallest angle; keep the cosine within -1..1 before arccos.
    # YOUR CODE HERE
    raise NotImplementedError("M51-T3: write your code here")

In [ ]:
st.check("M51-T3", func=kagan_angle)

Your function on the pairs of the prediction and of the Contract, and on the made-up case of Part 2: how far are
the best mechanism and the acceptable set from the made-up mechanism?

In [ ]:
for one, two in (((0.0, 45.0, -90.0), (0.0, 45.0, 90.0)), ((0.0, 45.0, -90.0), (0.0, 90.0, 0.0)),
                 ((0.0, 90.0, 0.0), (30.0, 90.0, 0.0)), ((0.0, 90.0, 0.0), (90.0, 90.0, 180.0))):
    print(f"{one} and {two}: {kagan_angle(one, two):.1f} deg")
p_angles = np.array([kagan_angle(TRUE_MECH, m) for m in p_acc])
print(f"\nPart 2, tol 0.05: best {kagan_angle(TRUE_MECH, p_best):.1f} deg from the made-up mechanism; the "
      f"{len(p_acc)} acceptable mechanisms lie {p_angles.min():.1f}-{p_angles.max():.1f} deg from it "
      f"(median {np.median(p_angles):.1f})")



**A set of fits has a width.** Each point is one of your acceptable mechanisms from the synthetic experiment.
Its horizontal coordinate is your Kagan angle to the known source; the vertical coordinate is its polarity
misfit. The histogram counts grid points, so its height is not a probability density for the true mechanism.

**Look:** does the smallest misfit also have the smallest angle to the truth? Why can a discrete search and
noisy polarities make those answers differ?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.5), constrained_layout=True)
axes[0].scatter(p_angles, p_acc_misfit, color="tab:blue", s=25, alpha=0.7)
axes[0].set(xlabel="Kagan angle to known source (°)", ylabel="Weighted polarity misfit", title="Your acceptable mechanisms")
axes[1].hist(p_angles, bins=np.linspace(0, 120, 13), color="tab:blue", edgecolor="white")
axes[1].set(xlabel="Kagan angle to known source (°)", ylabel="Number of acceptable grid points", title="Spread, not a posterior probability")
show_figure(fig, "M51_acceptable_set.png")

**Explain (3-5 sentences):** Use the printed lines. Was your prediction right for the two pairs, and which axes do
the two mechanisms of each pair share or swap? Why is the Kagan angle between (0, 90, 0) and (90, 90, 180) zero,
although no angle of the two triples agrees? How far was the best mechanism of Part 2 from the made-up one, and how
far did the acceptable set reach?

## Part 4 · Families of acceptable mechanisms (about 20 min, then task 4)

Sometimes the acceptable mechanisms do not form one cloud around the best one but two or more clouds far apart:
the polarities fit two quite different mechanisms about equally well, for example when the stations cover only
part of the focal sphere. Such clouds are **mechanism families** (glossary `mechanism_family`). A simple rule finds
them: sort the acceptable mechanisms by misfit; the first one starts a family and takes every mechanism within a
Kagan angle `max_angle` of it; the first of the mechanisms left starts the next family; and so on. SKHASH works in
the same spirit (Hardebeck & Shearer 2002; Skoumal et al. 2024): it averages the mechanisms within an angle of
45 degrees (its `cangle`) and reports a second solution when another group holds more than a fifth (its
`prob_max`) of the acceptable mechanisms.

### M51-T4 · Families of acceptable mechanisms · core · about 45 min

**Goal:** write `mechanism_families(accepted, misfit, kagan, max_angle)`: the families of a set of acceptable
mechanisms by the rule of Part 4.

**Given:** Part 4, your `grid_search` (task 2) to make sets to try it on, and a Kagan function (the check hands in
one of its own; in this notebook pass your `kagan_angle`). numpy and pandas.

**Contract:**
- Inputs: `accepted`, an array of shape (K, 3) of (strike, dip, rake) in any order; `misfit`, their K misfits;
  `kagan(mech1, mech2)`, a function that returns the Kagan angle in degrees; `max_angle` in degrees. Do not change
  the arrays.
- Output: a pandas DataFrame with the columns `strike`, `dip`, `rake`, `misfit`, `n`, `share`, one row per family
  in the order found: the family's first mechanism (its center: the remaining mechanism of lowest misfit, the first
  in the input among equal misfits) with its misfit, the number `n` of mechanisms in the family (the center
  included) and `share` = n / K. A mechanism joins a family when its Kagan angle from the family's center is at most
  `max_angle`.
- Example: mechanisms a, b, c in this order with misfits 0.10, 0.12, 0.11, where b lies 10 degrees from a, c 50
  degrees from a and 45 from b, and `max_angle = 30`: two rows, a (misfit 0.10, n 2, share 2/3) and c (0.11, 1,
  1/3).

**Done when** `st.check("M51-T4", func=mechanism_families)` passes on 12 made-up sets (two or three well separated
groups, ties of misfit, and chains of mechanisms that are each close to the next).

**Stuck?** `st.hint("M51-T4", 1)`

In [ ]:
# Before you code: three mechanisms lie along one line in rotation space, 0, 21 and 42 degrees from the first, with
# increasing misfits, and max_angle is 30. How many families will the rule of Part 4 make, and why is the answer not
# "one, because each mechanism is within 30 degrees of the next"? When would a set of polarities produce two
# families in the first place?
st.predict("M51-T4", """

""")

In [ ]:
def mechanism_families(accepted, misfit, kagan, max_angle):
    """Families of acceptable mechanisms: centers by misfit, members within max_angle (Kagan) of their center.

    accepted  : (K, 3) array of strike, dip, rake (degrees); misfit : their K misfits (both not changed)
    kagan     : function kagan(mech1, mech2) -> degrees; max_angle : degrees
    returns   : DataFrame with strike, dip, rake, misfit (of the center), n and share = n / K, in the order found
    """
    # Step 1: the positions of the mechanisms ordered by misfit (a stable sort keeps the input order among equals).
    # Step 2: while some are left: the first left is the center; its family is every left mechanism within
    #         max_angle of it; write one row and remove the family.
    # YOUR CODE HERE
    raise NotImplementedError("M51-T4: write your code here")

In [ ]:
st.check("M51-T4", func=mechanism_families)

Your two functions on made-up polarities that two quite different mechanisms explain equally well: 30 stations
kept only where the two mechanisms predict the same polarity.

In [ ]:
MECH_A, MECH_B = (120.0, 60.0, 45.0), (20.0, 70.0, -150.0)    # two made-up mechanisms
print(f"made-up mechanisms {MECH_A} and {MECH_B}: {kagan_angle(MECH_A, MECH_B):.1f} deg apart")
two_rng = np.random.default_rng(7)
t_az, t_inc = two_rng.uniform(0.0, 360.0, 400), two_rng.uniform(20.0, 160.0, 400)
pred_a = polarity_misfit(*MECH_A, t_az, t_inc, np.ones(400), np.ones(400))[1]
pred_b = polarity_misfit(*MECH_B, t_az, t_inc, np.ones(400), np.ones(400))[1]
keep = np.flatnonzero(pred_a == pred_b)[:30]
t_az, t_inc, t_pol, t_w = t_az[keep], t_inc[keep], pred_a[keep].astype(float), np.ones(keep.size)
t_best, t_best_misfit, t_acc, t_acc_misfit = grid_search(t_az, t_inc, t_pol, t_w, 10.0, 0.0, polarity_misfit)
t_fam = mechanism_families(t_acc, t_acc_misfit, kagan_angle, FAMILY_DEG)
print(f"{len(t_acc)} mechanisms fit all {keep.size} stations; their families at {FAMILY_DEG:g} deg:")
t_fam["from A (deg)"] = [kagan_angle(MECH_A, tuple(r)) for r in t_fam[["strike", "dip", "rake"]].to_numpy()]
t_fam["from B (deg)"] = [kagan_angle(MECH_B, tuple(r)) for r in t_fam[["strike", "dip", "rake"]].to_numpy()]
print(t_fam.round(2).to_string(index=False))

**Two different sources can explain the same polarities.** The observations were deliberately retained only
where the two synthetic mechanisms agree. The family shares count acceptable grid points under this search;
they are not independent evidence that one source is more likely.

**Look:** where would a new station need to lie to discriminate between these two sources? Explain why
adding more stations in the already sampled directions may do little.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
draw_mechanism(axes[0], MECH_A, "Synthetic source A")
draw_mechanism(axes[1], MECH_B, "Synthetic source B", color="tab:orange")
axes[2].bar(np.arange(1, len(t_fam) + 1), t_fam["share"], color="0.4")
axes[2].set(xlabel="Family number", ylabel="Share of acceptable grid points", ylim=(0, 1),
            title="Your acceptable-set families")
axes[2].set_xticks(np.arange(1, len(t_fam) + 1))
show_figure(fig, "M51_mechanism_families.png")

**Explain (3-5 sentences):** Use the printed table. How many families did your function find, and how close are
their centers to the two made-up mechanisms? Which family holds most of the acceptable mechanisms, and does a larger
share mean that its mechanism is more likely to be the true one here? What would one more station, placed where the
two mechanisms predict opposite polarities, do to the families?

## Part 5 · SKHASH through PocketQuake (about 40 min, then task 5)

PocketQuake's last stage, `focal_mechanism`, runs SKHASH on your cluster. It takes the HYPOINVERSE locations of the
chosen **velocity model** (glossary `velocity_model`; kim1983, Kim & Kim 1983) and the PhaseNet+ picks (Zhu et al.
2025; EQNet, Zhu et al. 2022): a P polarity from every P pick whose **pick probability** (glossary
`pick_probability`) is at least 0.5, and an **S/P amplitude ratio** (glossary `sp_amplitude_ratio`) from every
station with both picks whose amplitudes stand at least 3 times above the noise before P. S/P ratios add
information where polarities are scarce, because the ratio is small near the P lobes and large near the nodal
planes (Hardebeck & Shearer 2003). SKHASH traces rays in the model to get each takeoff angle, runs a grid search
like yours 30 times with the angles and the location shaken within their errors, and grades the result with a
**mechanism quality** (glossary `mechanism_quality`) from A (well fixed) to D, from the spread of the acceptable
mechanisms and the share of them near the reported one. PocketQuake keeps A and B as "high confidence".

The settings come from two files of the tools. The lines below are printed from your own copies, so what you read
is what runs.

In [ ]:
def show_lines(path, words):
    """Print the lines of a file that contain any of the words (with their line numbers)."""
    for number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        if any(w in line for w in words):
            print(f"{Path(path).name}:{number}: {line.strip()}")


TOOLS = st.pq.tools_env()
SKHASH_PY = Path(TOOLS.get("SKHASH_DIR", "SKHASH")) / "SKHASH.py"
show_lines(st.pq.pipeline_dir() / "pipeline" / "config.py",
           ["fm_velmodel:", "fm_min_polarity_weight:", "fm_min_pick_prob:", "fm_quality_keep:", "fm_use_sp_ratio:",
            "fm_sp_min_snr:", "fm_npolmin:", "fm_delmax_km:", "fm_max_agap:", "fm_max_pgap:", "cuspid_offset:"])
print()
show_lines(SKHASH_PY, ["'nmc':", "'badfrac':", "'badmin':", "'cangle':", "'prob_max':", "'dang':"])

This notebook's parameters follow these lines: `MIN_WEIGHT` = 0.3 is `fm_min_polarity_weight`, `FAMILY_DEG` = 45
is SKHASH's `cangle`, and a family share of 0.2 is its `prob_max`. Two differ on purpose: your grid step
`STEP_DEG` = 5 equals SKHASH's `dang` but SKHASH spaces its fault normals evenly on the sphere, and your `TOL` =
0.1 is a share of the weight above the best misfit, while SKHASH's `badfrac` = 0.1 sets a number of polarities
allowed to be wrong (at least `badmin` = 2). SKHASH also uses the S/P ratios; your search uses polarities only.

Your cluster first: it must hold the HYPOINVERSE locations of module M32 and PhaseNet+'s picks of module M21.

In [ ]:
sum_files = sorted((RUN_DIR / "1.HypoInv" / VELMODEL).glob("*.sum"))
pick_files = sorted((RUN_DIR / "picks").glob("*_picks.csv"))
if not sum_files or not pick_files:
    raise FileNotFoundError(f"PocketQuake has no HYPOINVERSE locations or picks of cluster {PQ_SLUG!r} under "
                            f"{RUN_DIR}. Part 5 starts from them: finish task M32-T1 of module M32 first (or set "
                            "PQ_SLUG to the cluster you located there).")
hypo = st.pq.read_sum(PQ_SLUG, VELMODEL)
print(f"HYPOINVERSE ({VELMODEL}) of cluster {PQ_SLUG!r}, from {hypo.attrs['source']}:")
print(hypo[["id", "time", "lat", "lon", "depth", "num", "gap", "rms", "qual"]].to_string(index=False))

The mechanism stage: a **dry run** (glossary `dry_run`) first, which only prints what would happen; read it, then
run the next cell (about a minute).

In [ ]:
st.pq.run(PQ_SLUG, "focal_mechanism", "focal_mechanism", cores=PQ_CORES, mem_gb=PQ_MEM_GB)      # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "focal_mechanism", "focal_mechanism", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0

Copy the stage's folder right away (a later run overwrites it) and take the **SHA-256 fingerprint** (glossary
`checksum`) of its mechanism file; sgtl logged the same fingerprint in its **run log** (glossary `run_log`),
`work/.sgtl/runs.jsonl` (module M41, Part 7). The table that follows names each event twice: `event_id` is
PocketQuake's name of the event (its origin time, UTC), and `cuspid` (glossary `cuspid`) is the whole number that
PocketQuake gives the same event in the files it writes for HYPOINVERSE, HypoDD and SKHASH (`cuspid_offset` plus
the event's place in the cluster's list; the config line is printed above). SKHASH's own tables (Skoumal et al.
2024), such as `OUT/out_polinfo.csv`, call that number `event_id`.

In [ ]:
def copy_run(name):
    """Copy the mechanism stage's folder to results/<name> (replacing an older copy); return the copy's path."""
    target = Path(RESULT_DIR) / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(FM_DIR, target)
    return target


def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


FIRST = copy_run("M51_skhash_first")
print("copied to", FIRST, "\nmechanisms.csv SHA-256:", sha256_of(FIRST / "mechanisms.csv"))
mech_sk = pd.read_csv(FIRST / "mechanisms.csv", dtype={"event_id": str})
print(mech_sk[["event_id", "cuspid", "strike", "dip", "rake", "quality", "num_p_pol", "num_sp_ratios",
               "polarity_misfit", "prob_mech", "mult_solution_flag", "magnitude"]].to_string(index=False))

SKHASH also writes, for every polarity it used, the station, its distance and the two angles it computed:
`OUT/out_polinfo.csv`. Your grid search can run on exactly these polarities and angles, so that the only
differences from SKHASH come from the search itself.

### M51-T5 · Your grid search on the Sangju polarities, next to SKHASH and KIGAM · core · about 90 min

**Goal:** write `skhash_polarities(polinfo, min_weight)`: the station arrays of task 1 from SKHASH's polarity
table. Then run your grid search on every Sangju event, compare your best mechanisms with SKHASH's and the
mainshock's with KIGAM's, and report it all in a dictionary `mech_report`.

**Given:** your copy `FIRST` of the first run (`OUT/out_polinfo.csv`, `mechanisms.csv`), SKHASH's line about its
takeoff angles (printed after the prediction), your functions of tasks 1-4, and KIGAM's mechanism of the
2019-07-21 ML 3.9 mainshock from its 2019 bulletin (KIGAM 2021, report KEMS-2019-062): nodal planes 288/89/7 and
198/83/179 (`KIGAM_MAINSHOCK` holds the first).

**Contract (the function):**
- Input: `polinfo`, a DataFrame with SKHASH's columns `event_id`, `sta_code` (NET.STA.LOC.CHA), `p_polarity`
  (signed: the sign is up or down, the size the weight; empty on S/P rows), `takeoff` and `azimuth` (degrees, as
  SKHASH writes them) and `source` (`trad_p` for a polarity, `trad_sp` for an S/P ratio), among others; `min_weight`.
  Do not change the table.
- Output: a DataFrame with the columns `event_id` (as in the table), `station` (NET.STA), `azimuth`, `takeoff`
  (degrees from the downward vertical, the convention of task 1), `polarity` (+1 or -1) and `weight` (the size of
  `p_polarity`), one row per P polarity whose size is at least `min_weight`, sorted by `event_id`, then `station`.

**Contract (the report):** `mech_report = {"skhash_polarities", "polarity_misfit", "grid_search", "kagan"`: your
four functions; `"slug"`: `PQ_SLUG`; `"folder"`: the path of `FIRST`; `"mechanisms_sha256"`: the SHA-256 of
`FIRST/mechanisms.csv`; `"step_deg"`, `"tol"`, `"min_weight"`: `STEP_DEG`, `TOL`, `MIN_WEIGHT`; `"best"`: {event id
of mechanisms.csv: your best (strike, dip, rake)}; `"kagan_skhash"`: {event id: Kagan angle between your best and
SKHASH's mechanism}; `"kagan_kigam"`: the Kagan angle between your best of the mainshock and `KIGAM_MAINSHOCK`}.
SKHASH's `event_id` in `out_polinfo.csv` is the `cuspid` of `mechanisms.csv`.

**Done when** `st.check("M51-T5", answer=mech_report)` passes: your function on 8 made-up SKHASH tables; then the
check recomputes the fingerprint, finds it in the run log, reruns your functions on your own table and recomputes
your Kagan angles with an independent library; for the events SKHASH grades A from at least 50 polarities, your
best must lie within 30 degrees of SKHASH's.

**Stuck?** `st.hint("M51-T5", 1)`

In [ ]:
# Before you code: a station 12 km from the epicenter of a source 15 km deep. Taking the ray as straight, what is its
# takeoff angle in the convention of task 1? What number would the same ray have in a convention where 0 is straight
# up and 180 straight down? Which of your functions would give a wrong mechanism if you mixed the two up, and would
# its misfit tell you?
st.predict("M51-T5", """

""")

Now read SKHASH's own line about its takeoff angles, and the three nearest stations of the mainshock (the largest
event of `mechanisms.csv`) in its polarity table:

In [ ]:
show_lines(SKHASH_PY, ["'flip_takeoff':"])
polinfo = pd.read_csv(FIRST / "OUT" / "out_polinfo.csv")
mainshock_cusp = int(mech_sk.loc[mech_sk["magnitude"].astype(float).idxmax(), "cuspid"])
nearest = polinfo[(polinfo["event_id"] == mainshock_cusp) & (polinfo["source"] == "trad_p")].nsmallest(3, "sr_dist_km")
print(nearest[["event_id", "sta_code", "p_polarity", "sr_dist_km", "origin_depth_km", "takeoff", "azimuth"]]
      .to_string(index=False))

In [ ]:
def skhash_polarities(polinfo, min_weight):
    """The station arrays of task 1 from SKHASH's polarity table (out_polinfo.csv).

    polinfo    : DataFrame with event_id, sta_code, p_polarity, takeoff, azimuth, source (not changed)
    min_weight : smallest |p_polarity| kept
    returns    : DataFrame with event_id, station (NET.STA), azimuth, takeoff (from the downward vertical),
                 polarity (+1/-1) and weight, sorted by event_id and station
    """
    # Step 1: the P polarity rows whose size reaches min_weight.
    # Step 2: station = sta_code up to its second dot; takeoff turned into the convention of task 1.
    # YOUR CODE HERE
    raise NotImplementedError("M51-T5: write your code here")

Your functions on every event of your run (about 10 s; nothing is printed until the check has run):

In [ ]:
names = dict(zip(mech_sk["cuspid"].astype(int), mech_sk["event_id"]))
MAINSHOCK = names[mainshock_cusp]
rays = skhash_polarities(polinfo, MIN_WEIGHT)
searched = {}
for cusp, rows in rays.groupby("event_id"):
    searched[names[int(cusp)]] = grid_search(*(rows[c].to_numpy(float) for c in ("azimuth", "takeoff", "polarity",
                                                                                     "weight")),
                                             STEP_DEG, TOL, polarity_misfit)
best = {event: found[0] for event, found in searched.items()}
skhash_mech = {event: tuple(mech_sk.loc[mech_sk["event_id"] == event, ["strike", "dip", "rake"]].iloc[0])
               for event in best}
mech_report = {"skhash_polarities": skhash_polarities, "polarity_misfit": polarity_misfit,
               "grid_search": grid_search, "kagan": kagan_angle, "slug": PQ_SLUG, "folder": str(FIRST),
               "mechanisms_sha256": sha256_of(FIRST / "mechanisms.csv"), "step_deg": STEP_DEG, "tol": TOL,
               "min_weight": MIN_WEIGHT, "best": best,
               "kagan_skhash": {event: kagan_angle(best[event], skhash_mech[event]) for event in best},
               "kagan_kigam": kagan_angle(best[MAINSHOCK], KIGAM_MAINSHOCK)}

In [ ]:
st.check("M51-T5", answer=mech_report)

Your mechanisms next to SKHASH's, and the families of your acceptable mechanisms (task 4, `FAMILY_DEG`):

In [ ]:
table = mech_sk.drop_duplicates("event_id")[["event_id", "quality", "num_p_pol", "strike", "dip", "rake"]].copy()
table["yours"] = table["event_id"].map(lambda e: "/".join(f"{v:g}" for v in best[e]))
table["your misfit"] = table["event_id"].map(lambda e: round(searched[e][1], 3))
table["Kagan to SKHASH"] = table["event_id"].map(lambda e: round(mech_report["kagan_skhash"][e], 1))
print(table.to_string(index=False))
print(f"\nmainshock {MAINSHOCK}: your best is {mech_report['kagan_kigam']:.1f} deg from KIGAM's mechanism, "
      f"SKHASH's {kagan_angle(skhash_mech[MAINSHOCK], KIGAM_MAINSHOCK):.1f} deg; KIGAM's two planes give "
      f"{kagan_angle(KIGAM_MAINSHOCK, (198.0, 83.0, 179.0)):.1f} deg between them")
families = {}
for event, found in searched.items():
    families[event] = mechanism_families(found[2], found[3], kagan_angle, FAMILY_DEG)
    top = families[event].head(3)
    shown = "; ".join(f"{r.strike:g}/{r.dip:g}/{r.rake:g} share {r.share:.2f}" for r in top.itertuples())
    print(f"{event}: {len(found[2])} acceptable, {len(families[event])} families: {shown}")

**Explain (3-5 sentences):** Use the printed tables. For which events does your best mechanism lie close to SKHASH's
and for which far, and how do the number of polarities and SKHASH's grade go with that? How far are your mainshock
mechanism and SKHASH's from KIGAM's, and what does that size of difference mean for a mechanism of an ML 3.9? Which
events have a second family that holds more than a fifth of their acceptable mechanisms? Read the rows of the
2022-07-06 event (the last row of the table and its line of families): which grade did SKHASH give it, how many
families did your search find, and what do the two together say about how well its first motions fix one
mechanism? Say what your search leaves out that SKHASH includes.

**Compare the mainshock mechanisms in the same projection.** These are your grid-search result, SKHASH's
result and the published KIGAM mechanism. First motions leave two nodal planes; visual agreement of beachballs
does not select the fault that actually slipped. Use your computed Kagan angles to quantify the comparison.

**Look:** which parts of the compressional quadrants move between solutions? Relate their agreement or
disagreement to station coverage and to the differences between the three estimation methods.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
for ax, mechanism, label in zip(axes, (best[MAINSHOCK], skhash_mech[MAINSHOCK], KIGAM_MAINSHOCK),
                                 ("Your polarity grid", "SKHASH", "KIGAM bulletin")):
    draw_mechanism(ax, mechanism, label)
fig.suptitle("Sangju 2019 mainshock: lower-hemisphere P radiation", fontsize=11)
show_figure(fig, "M51_mainshock_comparison.png")

Optional explorer E8 (not checked): the mainshock's polarities on the focal sphere. Move strike, dip and rake, or
jump to SKHASH's, KIGAM's or your own mechanism; the shading, the misfit and the stations marked in red all come
from your `polarity_misfit`. The explorer places the stations with your `equal_area` of module M50 (task 4):
paste it into the next cell and run it (without it you see the rim of the net and the misfit only).

In [ ]:
# Paste your equal_area of module M50 below this line.

In [ ]:
main_rays = rays[rays["event_id"] == mainshock_cusp]
your_net = globals().get("equal_area")          # your M50 function, if you pasted it above
st.widgets.beachball_explorer(main_rays["azimuth"], main_rays["takeoff"], main_rays["polarity"], main_rays["weight"],
                              project=your_net, misfit=polarity_misfit, start=best[MAINSHOCK],
                              stations=list(main_rays["station"]),
                              marks={"SKHASH": skhash_mech[MAINSHOCK], "KIGAM": KIGAM_MAINSHOCK,
                                     "your grid search": best[MAINSHOCK]})

## Part 6 · An unchanged rerun, the S/P ratios, and polarities only (about 90 min, task 6)

PocketQuake takes each pick's amplitude from PhaseNet+ (EQNet): the largest absolute value, over the three
components, in an **amplitude window** (glossary `amplitude_window`) that starts at the pick. The S/P ratio of a
station is its S amplitude divided by its P amplitude. The **S-P time** (glossary `s_minus_p_time`) of a station is
its S pick time minus its P pick time; it grows by about 1 s for every 8 km of distance at these depths.

Task 6 runs the stage twice more: first unchanged (does SKHASH give the same bytes again? It is a
**deterministic computation**, glossary `deterministic_computation`, only if its random draws start from a fixed
seed), then with one setting changed: PocketQuake's `fm_use_sp_ratio`, from `True` to `False`, so that SKHASH sees
polarities only. The setting goes into your cluster module (the file `st.pq.stage` wrote in module M21) as one
extra line; the cell puts the file back as it was right after the run.

### M51-T6 · An unchanged rerun, the S/P ratios, and polarities only · core · about 90 min

**Goal:** write `sp_table(amp, picks)`: the S/P ratio and the S-P time of every station. Then rerun the mechanism
stage unchanged and with polarities only, and report the runs in a dictionary `sp_report`. After the check you
look at how the S/P ratios that PocketQuake handed to SKHASH spread against the S-P time.

**Given:** the run cells below (read each one before you run it), your copy `FIRST` (`IN/amp.csv`, the S/P ratios
of your first run), your cluster's pick files and your `kagan_angle` (task 3).

**Contract (the function):**
- Inputs: `amp`, a DataFrame with the columns `event_id`, `network`, `station` and `sp_ratio` (one row per station
  and event, maybe more columns); `picks`, a DataFrame with `event_id`, `network`, `station`, `phase` (`"P"` or
  `"S"`) and `time` (pandas Timestamps), maybe several picks of one phase at a station. Do not change them.
- Output: a DataFrame with the columns `event_id`, `network`, `station`, `sp_ratio` (as in `amp`) and `sp_time_s`
  (the earliest S pick minus the earliest P pick of that event and station, in seconds; NaN when either is
  missing), one row per row of `amp`, sorted by `event_id`, `network`, `station`.

**Contract (the report):** `sp_report = {"sp_table": sp_table, "slug": PQ_SLUG, "first": FIRST, "rerun": RERUN,
"polarities_only": POL_ONLY` (the paths of the three copies), `"mechanisms_sha256"`: {"first", "rerun",
"polarities_only": the SHA-256 of each copy's `mechanisms.csv`}, `"kagan_without_sp"`: {event id of
`mechanisms.csv`: the Kagan angle between the event's mechanism in `FIRST` and in `POL_ONLY` (the first row of the
event in each file), for every event of `FIRST` that `POL_ONLY` also has}}.

**Done when** `st.check("M51-T6", answer=sp_report)` passes: your function on 8 made-up tables; then the check
recomputes the three fingerprints, finds them in the run log, compares the control files and inputs of the three
runs (exactly one change) and recomputes your Kagan angles between the two runs with an independent library.

**Stuck?** `st.hint("M51-T6", 1)`

In [ ]:
# Before you code: picture the S/P ratios of your first run plotted against the S-P time of their station. Which
# values do you expect them to take, how widely should they spread at one S-P time, and should that spread change
# with the S-P time, and how? What will happen to SKHASH's quality grades and mechanisms when it gets
# polarities only?
st.predict("M51-T6", """

""")

In [ ]:
def sp_table(amp, picks):
    """S/P ratios with the S-P time of their station.

    amp   : DataFrame with event_id, network, station, sp_ratio (not changed)
    picks : DataFrame with event_id, network, station, phase ("P"/"S") and time (Timestamps) (not changed)
    returns : DataFrame with event_id, network, station, sp_ratio, sp_time_s, sorted
    """
    # Step 1: the earliest pick of each phase per event, network and station.
    # Step 2: S minus P in seconds, joined to amp so that every row of amp stays (NaN without both picks).
    # YOUR CODE HERE
    raise NotImplementedError("M51-T6: write your code here")

**The unchanged rerun** (about a minute), copied into its own folder:

In [ ]:
assert st.pq.run(PQ_SLUG, "focal_mechanism", "focal_mechanism", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
RERUN = copy_run("M51_skhash_rerun")
print("rerun copied to", RERUN)

**The one change.** Your cluster module ends with PocketQuake's own settings line; the next cell adds one line,
runs the stage (a few seconds: SKHASH is much lighter without the ratios), copies the folder and puts the module
back exactly as it was (its fingerprint is compared). It keeps the run's printout for later: the full output is in
the log file named on the last line, and you read the results after the check.

In [ ]:
CLUSTER_PY = st.pq.pipeline_dir() / "pipeline" / "clusters" / f"{PQ_SLUG}.py"
module_text = CLUSTER_PY.read_text(encoding="utf-8")
CHANGE = "CONFIG = replace(CONFIG, fm_use_sp_ratio=False)   # M51: polarities only\n"
print("last line of your cluster module now:", module_text.strip().splitlines()[-1])
print("the line this cell adds for one run:  ", CHANGE.strip())
quiet = io.StringIO()
try:
    CLUSTER_PY.write_text(module_text + "\n" + CHANGE, encoding="utf-8")
    with contextlib.redirect_stdout(quiet):
        changed_rc = st.pq.run(PQ_SLUG, "focal_mechanism", "focal_mechanism", cores=PQ_CORES, mem_gb=PQ_MEM_GB,
                               apply=True)
finally:
    CLUSTER_PY.write_text(module_text, encoding="utf-8")
print(quiet.getvalue().strip().splitlines()[-1])
assert changed_rc == 0, quiet.getvalue()[-2000:]
POL_ONLY = copy_run("M51_skhash_polarities_only")
print("copied to", POL_ONLY)
print("cluster module restored:", hashlib.sha256(CLUSTER_PY.read_bytes()).hexdigest()
      == hashlib.sha256(module_text.encode("utf-8")).hexdigest())

The report (nothing is printed until the check has run):

In [ ]:
mech_pol = pd.read_csv(POL_ONLY / "mechanisms.csv", dtype={"event_id": str})
first_rows = mech_sk.drop_duplicates("event_id").set_index("event_id")
pol_rows = mech_pol.drop_duplicates("event_id").set_index("event_id")
sp_report = {"sp_table": sp_table, "slug": PQ_SLUG, "first": str(FIRST), "rerun": str(RERUN),
             "polarities_only": str(POL_ONLY),
             "mechanisms_sha256": {key: sha256_of(Path(folder) / "mechanisms.csv")
                                   for key, folder in (("first", FIRST), ("rerun", RERUN),
                                                       ("polarities_only", POL_ONLY))},
             "kagan_without_sp": {event: kagan_angle(tuple(first_rows.loc[event, ["strike", "dip", "rake"]]),
                                                     tuple(pol_rows.loc[event, ["strike", "dip", "rake"]]))
                                  for event in first_rows.index if event in pol_rows.index}}

In [ ]:
st.check("M51-T6", answer=sp_report)

Now measure. First the rerun: the same bytes as the first run? Then how the S/P ratios spread against the S-P time
(your `sp_table` on your first run's ratios and your cluster's picks): the values that occur most often, the
smallest, median and largest ratio in bins of S-P time, and the plot of every ratio against its S-P time.

In [ ]:
print("rerun's mechanisms.csv the same bytes as the first run's:",
      sha256_of(RERUN / "mechanisms.csv") == sha256_of(FIRST / "mechanisms.csv"))
print("the random seed in SKHASH's code:")
show_lines(Path(TOOLS.get("SKHASH_DIR", "SKHASH")) / "functions" / "fun.py", ["default_rng("])
print()
amp = pd.read_csv(FIRST / "IN" / "amp.csv")
cusp_of = {event: cusp for cusp, event in names.items()}
all_picks = pd.concat([pd.read_csv(f, dtype={"Event_ID": str}) for f in pick_files], ignore_index=True)
picks = pd.DataFrame({"event_id": all_picks["Event_ID"].map(cusp_of), "network": all_picks["Network"],
                      "station": all_picks["Station"], "phase": all_picks["Phase"],
                      "time": pd.to_datetime(all_picks["Time"], utc=True)}).dropna(subset=["event_id"])
picks["event_id"] = picks["event_id"].astype(int)
ratios = sp_table(amp, picks)
print(f"{len(ratios)} S/P ratios, {int(ratios['sp_time_s'].notna().sum())} with an S-P time; "
      f"{ratios['sp_ratio'].nunique()} different values. The five most common values (value: count):")
print(ratios["sp_ratio"].value_counts().head(5).to_string())
top = ratios["sp_ratio"].value_counts().index[0]
at_top = ratios["sp_ratio"] == top
print(f"the most common value {top!r}: {int(at_top.sum())} of {len(ratios)} ratios ({100 * at_top.mean():.1f} %), "
      f"at S-P times up to {ratios.loc[at_top, 'sp_time_s'].max():.2f} s")
bins = pd.cut(ratios["sp_time_s"], [0, 4, 6, 8, 9, 10, 12, 16])
summary = ratios.groupby(bins, observed=True)["sp_ratio"].agg(["size", "min", "median", "max"])
summary.columns = ["stations", "smallest", "median", "largest"]
print(summary.round(3).to_string())
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for flag, colour, label in ((True, "tab:red", f"The most common value ({top:g})"), (False, "tab:blue", "Other")):
    part = ratios[at_top == flag]
    ax.plot(part["sp_time_s"], part["sp_ratio"], "o", ms=4, color=colour, label=label)
ax.set_xlabel("S-P time (s)")
ax.set_ylabel("S/P amplitude ratio")
ax.set_yscale("log")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)
fig.savefig(Path(FIGURE_DIR) / "sp_ratios.png", dpi=150, bbox_inches="tight")
plt.show()

The lines of EQNet's code that set the amplitude windows (your copy of the tools), and the mechanisms with and
without the ratios:

In [ ]:
show_lines(Path(TOOLS.get("EQNET_DIR", "EQNet")) / "eqnet" / "utils" / "postprocess.py",
           ["window_amp=[10, 5]", "window_amp_i = int(", "min(j1 + window_amp_i", 'pick_dict["phase_amplitude"]'])
rows = []
for event in mech_sk["event_id"].drop_duplicates():
    with_sp = mech_sk[mech_sk["event_id"] == event].iloc[0]
    without = mech_pol[mech_pol["event_id"] == event]
    if without.empty:
        rows.append({"event_id": event, "with S/P": f"{with_sp.strike:g}/{with_sp.dip:g}/{with_sp.rake:g}",
                     "grade": with_sp.quality, "polarities only": "none"})
        continue
    rows.append({"event_id": event, "with S/P": f"{with_sp.strike:g}/{with_sp.dip:g}/{with_sp.rake:g}",
                 "grade": with_sp.quality, "polarities only": "; ".join(f"{r.strike:g}/{r.dip:g}/{r.rake:g}"
                                                                         for r in without.itertuples()),
                 "grades": "".join(without["quality"]), "Kagan (deg)": round(sp_report["kagan_without_sp"][event], 1),
                 "your best to polarities-only": round(kagan_angle(best[event], tuple(
                     without[["strike", "dip", "rake"]].iloc[0])), 1)})
print(pd.DataFrame(rows).to_string(index=False))

**Explain (3-5 sentences):** Use the printed lines. Was the unchanged rerun the same bytes, and why (the seed line)?
Which value did the S/P ratios take most often, what share of them, and up to which S-P time? How do EQNet's window
lengths explain these numbers (look at what lies inside a P window of that length at these distances), and what
does such a ratio say about the radiation pattern? How did the grades and mechanisms change with polarities only,
and can these runs tell whether the ratios of the most common value or the others held the grades up? Is your own
polarity-only grid search closer to SKHASH with or without the ratios?

## Wrap-up

- `results/M51_skhash_first/`, `results/M51_skhash_rerun/`, `results/M51_skhash_polarities_only/`: your copies of
  the three runs of the mechanism stage; `work/.sgtl/runs.jsonl` logs every run with the fingerprint of its
  mechanism file.
- `figures/sp_ratios.png`: the S/P ratios against the S-P time.

The mechanisms rest on PhaseNet+'s polarities of the raw counts; no instrument response was needed (the course's
station metadata, StationMetadata 1.0.2 by Byeong-Seok Ahn, Pukyong National University, reference key `ahn2024`,
served modules M03 and M10). KIGAM's mechanism is quoted from its 2019 bulletin (KIGAM 2021). The KMA catalog is
used under the Korea Open Government License Type 1 (attribution).

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M51")

### Going further (optional, not checked)

1. Rerun your grid search on the 2022 event with half of its stations left out at random, ten times. Read the
   families of that event in your Part 5 printout first: how often does the best mechanism land in each of them?
2. Repeat Part 5 with `STEP_DEG = 10`. Which numbers of the tables change, and by how much?
3. Compute the takeoff angles of the mainshock's stations yourself with your travel-time functions of module M30
   (direct rays in kim1983) and compare them with SKHASH's. Where do they differ, and why (look at the stations
   beyond the crossover distance)?

Next: module M52 computes waveforms of these sources, and M53 fits moment tensors to the Sangju records.